In [ ]:


# --- 1. FILE PATHS & PARSING ---
var_char_file = 'Var_char_Fmin_Tese_Varactor_characterization_tb2_Measurements_history_1_20260416_16_39_25.18.csv'

# FIX: Use the 'tb2' file here since we need the Temperature sweep for df_prop
prop_var_file = 'PROP_INT_VAR_SIZING_COMP_Tese_Varactor_characterization_tb2_Measurements_history_1_20260609_14_30_56.90.csv'

# Updated to use the tb1 file which contains the CV curve (Capacitance vs Vdc)
comp_var_cv_file = 'VAR_COMP_SIZING_Tese_Varactor_characterization_tb1_Measurements_history_1_20260609_16_15_03.47.csv'

unit_map = {'f': 1e-15, 'p': 1e-12, 'n': 1e-9, 'u': 1e-6, 'm': 1e-3, 'k': 1e3, 'M': 1e6, 'G': 1e9}

def parse_units(value):
    if pd.isna(value) or str(value).strip().lower() == "error" or str(value).strip() == "":
        return np.nan
    val_str = str(value).strip()
    match = re.search(r'([0-9\.-]+)([a-zA-Z]*)', val_str)
    if match:
        num = float(match.group(1))
        multiplier = unit_map.get(match.group(2), 1.0)
        return num * multiplier
    return np.nan

# --- 2. LOAD DATA ---
df_char = pd.read_csv(var_char_file)
df_prop = pd.read_csv(prop_var_file)
df_cv = pd.read_csv(comp_var_cv_file)

# Parse Capacitance and Temp for the typical uncompensated curve
df_char['Cap_fF'] = df_char['Cap_measuredAt1Mhz_M7:ac'].apply(parse_units) * 1e15
df_char['Temp'] = pd.to_numeric(df_char['Sweep:Variable:temp'], errors='coerce')

# Parse real Capacitance and Temp for the Proportional Varactor
# FIX: Multiply by 2 here to represent Proportional + Integral varactor
df_prop['C_prop_fF'] = df_prop['Cap_max_measured:ac'].apply(parse_units) * 1e15 * 2
df_prop['Temp'] = pd.to_numeric(df_prop['Sweep:Variable:temp'], errors='coerce')

# Parse the CV curve (Vdc vs Capacitance) for the Compensation Varactor
df_cv['C_comp_fF'] = df_cv['Cap_max_measured:ac'].apply(parse_units) * 1e15
df_cv = df_cv.dropna(subset=['C_comp_fF', 'Sweep:Variable:Vdc'])

# Sort data carefully for interpolation
df_cv_v_sorted = df_cv.sort_values('Sweep:Variable:Vdc')
df_cv_c_sorted = df_cv.sort_values('C_comp_fF')

# Create interpolation functions mapping both ways
# Fix: Prevent wild extrapolation by clamping to boundary values instead
cv_interp = interp1d(
    df_cv_c_sorted['C_comp_fF'], 
    df_cv_c_sorted['Sweep:Variable:Vdc'], 
    bounds_error=False, 
    fill_value=(df_cv_c_sorted['Sweep:Variable:Vdc'].iloc[0], df_cv_c_sorted['Sweep:Variable:Vdc'].iloc[-1])
)

vc_interp = interp1d(
    df_cv_v_sorted['Sweep:Variable:Vdc'], 
    df_cv_v_sorted['C_comp_fF'], 
    bounds_error=False, 
    fill_value=(df_cv_v_sorted['C_comp_fF'].iloc[0], df_cv_v_sorted['C_comp_fF'].iloc[-1])
)

# Isolate Typical Corner (TT25)
df_tt = df_char[df_char['Corner'] == 'TT25'].sort_values('Temp').copy()

# Merge the real C_prop into our main dataframe aligning by Temperature
df_tt = df_tt.merge(df_prop[['Temp', 'C_prop_fF']], on='Temp', how='left')

# --- 3. MIRROR, SCALE, AND SATURATE DATA ---
# Mirror the capacitance values (reverse the array to invert the temperature dependence)
# Divide by 1.2 to scale it below the typical curve
df_tt['total C'] = df_tt['Cap_fF'].values[::-1] / 1.2

# Calculate the required Vdc to hit the target Total C
df_tt['C_comp_needed'] = df_tt['total C'] - df_tt['C_prop_fF']
df_tt['Vdc_mapped'] = cv_interp(df_tt['C_comp_needed'])

# SATURATION LOGIC: Clamp Vdc between 0.6V and 0.9V
df_tt['Vdc_actual'] = df_tt['Vdc_mapped'].clip(lower=0.6, upper=0.9)

# Calculate the actual capacitance provided and the resulting total capacitance
df_tt['C_comp_actual'] = vc_interp(df_tt['Vdc_actual'])
df_tt['Total_C_achieved'] = df_tt['C_prop_fF'] + df_tt['C_comp_actual']

# --- 4. PLOTTING ---
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(df_tt['Temp'], df_tt['Cap_fF'], label='Typical CT Curve (TT25)', marker='s', color='orange', linewidth=2)
ax.plot(df_tt['Temp'], df_tt['total C'], label='Target Total C (Mirrored / 1.2)', marker='o', color='purple', linewidth=2, linestyle='--')
# New curve showing the reality of the saturated Vdc mapping
ax.plot(df_tt['Temp'], df_tt['Total_C_achieved'], label='Achieved Total C (Saturated 0.6V-0.9V)', marker='^', color='green', linewidth=2)

ax.set_title('Capacitance vs Temperature (with Vdc Saturation Limits)', fontweight='bold')
ax.set_xlabel('Temperature (°C)')
ax.set_ylabel('Capacitance (fF)')
ax.grid(True, linestyle='--', alpha=0.7)
ax.legend(loc='best')

plt.tight_layout()
plt.show()

# --- 5. PRINT COMPENSATION CAPACITANCE TABLE ---
print("\n" + "="*115)
print(" CAPACITANCE TRACKING (Target vs Achieved with 0.6V - 0.9V Saturation)")
print("="*115)
print(f"{'Temp (°C)':<10} | {'C_prop+int':<12} | {'Needed C_comp':<15} | {'Ideal Vdc(V)':<14} | {'Actual Vdc(V)':<15} | {'Achieved Total C(fF)':<20}")
print("-" * 115)

for i in range(len(df_tt)):
    t = df_tt['Temp'].iloc[i]
    c_prop_real = df_tt['C_prop_fF'].iloc[i] 
    c_comp_needed = df_tt['C_comp_needed'].iloc[i]
    vdc_mapped = df_tt['Vdc_mapped'].iloc[i]
    vdc_actual = df_tt['Vdc_actual'].iloc[i]
    total_c_achieved = df_tt['Total_C_achieved'].iloc[i]
    
    print(f"{t:<10.1f} | {c_prop_real:<12.3f} | {c_comp_needed:<15.3f} | {vdc_mapped:<14.3f} | {vdc_actual:<15.3f} | {total_c_achieved:<20.3f}")

print("="*115 + "\n")

In [ ]:


# --- 1. CONFIGURATION & FILE PATHS ---
files = {
    # Using the newly uploaded files for the Proportional + Integral varactor
    'prop_tb1': 'PROP_INT_VAR_SIZING_COMP_Tese_Varactor_characterization_tb1_Measurements_history_1_20260609_14_30_56.90.csv',
    'prop_tb2': 'PROP_INT_VAR_SIZING_COMP_Tese_Varactor_characterization_tb2_Measurements_history_1_20260609_14_30_56.90.csv',
    'comp_tb1': 'VAR_COMP_SIZING_Tese_Varactor_characterization_tb1_Measurements_history_1_20260609_16_15_03.47.csv',
    'comp_tb2': 'VAR_COMP_SIZING_Tese_Varactor_characterization_tb2_Measurements_history_1_20260609_16_15_03.47.csv',
    'typ_ct': 'Var_char_Fmin_Tese_Varactor_characterization_tb2_Measurements_history_1_20260416_16_39_25.18.csv'
}

# --- 2. UNIT PARSING ---
unit_map = {'f': 1e-15, 'p': 1e-12, 'n': 1e-9, 'u': 1e-6, 'm': 1e-3, 'k': 1e3, 'M': 1e6, 'G': 1e9}

def parse_units(value):
    if pd.isna(value) or str(value).strip().lower() == "error" or str(value).strip() == "":
        return np.nan
    val_str = str(value).strip()
    match = re.search(r'([0-9\.-]+)([a-zA-Z]*)', val_str)
    if match:
        num = float(match.group(1))
        multiplier = unit_map.get(match.group(2), 1.0)
        return num * multiplier
    return np.nan

# --- 3. LOAD DATA ---
dfs = {}
for key, path in files.items():
    if os.path.exists(path):
        dfs[key] = pd.read_csv(path)
    else:
        print(f"Warning: File not found -> {path}")
        dfs[key] = pd.DataFrame()

def get_cap_ff(df, col='Cap_max_measured:ac'):
    if df.empty or col not in df.columns:
        return None
    return df[col].apply(parse_units) * 1e15

cap_col = 'Cap_max_measured:ac'

# --- 3B. EXTRACT Vdc_opt FOR BOTH VARACTORS (Near 25°C) ---
vdc_opt_prop_mark = None
if not dfs['prop_tb2'].empty and 'Vdc_opt:variable' in dfs['prop_tb2'].columns and 'Sweep:Variable:temp' in dfs['prop_tb2'].columns:
    df_temp = dfs['prop_tb2'].copy()
    df_temp['temp_diff'] = abs(df_temp['Sweep:Variable:temp'] - 25)
    closest_idx = df_temp['temp_diff'].idxmin()
    vdc_opt_prop_mark = parse_units(df_temp.loc[closest_idx, 'Vdc_opt:variable'])

vdc_opt_comp_mark = None
if not dfs['comp_tb2'].empty and 'Vdc_opt:variable' in dfs['comp_tb2'].columns and 'Sweep:Variable:temp' in dfs['comp_tb2'].columns:
    df_temp2 = dfs['comp_tb2'].copy()
    df_temp2['temp_diff'] = abs(df_temp2['Sweep:Variable:temp'] - 25)
    closest_idx2 = df_temp2['temp_diff'].idxmin()
    vdc_opt_comp_mark = parse_units(df_temp2.loc[closest_idx2, 'Vdc_opt:variable'])

# ==========================================
# FIGURE 1: CV Curves (Vdc_opt Marks)
# ==========================================
fig1, (ax1_prop, ax1_comp) = plt.subplots(1, 2, figsize=(12, 5))
fig1.suptitle('CV Curves: Proportional + Integral vs Compensation', fontsize=14, fontweight='bold')

# Left: Proportional + Integral CV
if not dfs['prop_tb1'].empty and cap_col in dfs['prop_tb1'].columns:
    v_prop_cv = dfs['prop_tb1']['Sweep:Variable:Vdc']
    
    # MULTIPLYING BY 2 FOR PROPORTIONAL + INTEGRAL
    c_prop_cv = get_cap_ff(dfs['prop_tb1']) * 2 
    
    ax1_prop.plot(v_prop_cv, c_prop_cv, color='red', linewidth=2, marker='o', label='CV Curve')
    
    if vdc_opt_prop_mark is not None:
        f_cv_prop = interp1d(v_prop_cv, c_prop_cv, bounds_error=False)
        c_opt_prop = f_cv_prop(vdc_opt_prop_mark)
        ax1_prop.axvline(vdc_opt_prop_mark, color='black', linestyle='--', alpha=0.6, label=f'Vdc_opt ({vdc_opt_prop_mark*1000:.0f} mV)')
        if not np.isnan(c_opt_prop):
            ax1_prop.plot(vdc_opt_prop_mark, c_opt_prop, marker='*', markersize=14, color='gold', markeredgecolor='black', label=f'C_opt ≈ {c_opt_prop:.1f} fF')
        ax1_prop.legend(loc='best')
        
    ax1_prop.set_title('Proportional + Integral Varactor')
    ax1_prop.set_xlabel('Voltage (V)')
    ax1_prop.set_ylabel('Capacitance (fF)')
    ax1_prop.grid(True, linestyle='--', alpha=0.6)

# Right: Compensation CV (Un-inverted)
if not dfs['comp_tb1'].empty and cap_col in dfs['comp_tb1'].columns:
    c_comp_cv = get_cap_ff(dfs['comp_tb1'])
    v_orig = dfs['comp_tb1']['Sweep:Variable:Vdc'].apply(parse_units)
    plot_df = pd.DataFrame({'V': v_orig, 'C': c_comp_cv}).dropna().sort_values('V')
    ax1_comp.plot(plot_df['V'], plot_df['C'], color='blue', linewidth=2, marker='o', label='CV Curve')
    
    # Updated fixed physical Vdc limits to 0.6V and 0.9V
    ax1_comp.axvline(0.6, color='gray', linestyle='-.', alpha=0.8, label='Vdc = 0.6 V')
    ax1_comp.axvline(0.9, color='dimgray', linestyle='-.', alpha=0.8, label='Vdc = 0.9 V')
    
    if vdc_opt_comp_mark is not None:
        f_cv_comp = interp1d(plot_df['V'], plot_df['C'], bounds_error=False)
        c_opt_comp = f_cv_comp(vdc_opt_comp_mark)
        
        ax1_comp.axvline(vdc_opt_comp_mark, color='black', linestyle='--', alpha=0.6, label=f'Actual Vdc_opt ({vdc_opt_comp_mark*1000:.0f} mV)')
        if not np.isnan(c_opt_comp):
            ax1_comp.plot(vdc_opt_comp_mark, c_opt_comp, marker='*', markersize=14, color='gold', markeredgecolor='black', label=f'C_opt ≈ {c_opt_comp:.1f} fF')
        ax1_comp.legend(loc='best')
    else:
        ax1_comp.legend(loc='best')

    ax1_comp.set_title('Compensation Varactor')
    ax1_comp.set_xlabel('Voltage (V)')
    ax1_comp.set_ylabel('Capacitance (fF)')
    ax1_comp.grid(True, linestyle='--', alpha=0.6)
fig1.tight_layout()

# ==========================================
# FIGURE 2: CT Curves (Text Boxes for Both)
# ==========================================
fig2, (ax2_prop, ax2_comp) = plt.subplots(1, 2, figsize=(12, 5))
fig2.suptitle('CT Curves: Proportional + Integral vs Compensation', fontsize=14, fontweight='bold')

# Left: Proportional + Integral CT
if not dfs['prop_tb2'].empty and cap_col in dfs['prop_tb2'].columns:
    temps_prop = dfs['prop_tb2']['Sweep:Variable:temp']
    
    # MULTIPLYING BY 2 FOR PROPORTIONAL + INTEGRAL
    ct_prop = get_cap_ff(dfs['prop_tb2']) * 2
    
    ax2_prop.plot(temps_prop, ct_prop, color='orange', linewidth=2, marker='o', label='Capacitance')
    ax2_prop.set_title('Proportional + Integral Varactor')
    ax2_prop.set_xlabel('Temperature (°C)')
    ax2_prop.set_ylabel('Capacitance (fF)')
    ax2_prop.grid(True, linestyle='--', alpha=0.6)
    
    # Indicate fixed Vdc_opt for Proportional with a text box
    if vdc_opt_prop_mark is not None:
        fixed_vdc_text_prop = f"Vdc_opt Fixed @ {vdc_opt_prop_mark*1000:.0f} mV"
        ax2_prop.text(0.05, 0.95, fixed_vdc_text_prop, transform=ax2_prop.transAxes, 
                      fontsize=11, fontweight='bold', color='darkred', verticalalignment='top', 
                      bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='darkred', alpha=0.9))
        ax2_prop.legend(loc='lower right')

# Right: Compensation CT (Un-inverted)
if not dfs['comp_tb2'].empty and cap_col in dfs['comp_tb2'].columns:
    ct_comp = get_cap_ff(dfs['comp_tb2'])
    t_orig = dfs['comp_tb2']['Sweep:Variable:temp']
    plot_df2 = pd.DataFrame({'T': t_orig, 'C': ct_comp}).sort_values('T')
    
    ax2_comp.plot(plot_df2['T'], plot_df2['C'], color='purple', linewidth=2, marker='o', label='Capacitance')
    ax2_comp.set_title('Compensation Varactor')
    ax2_comp.set_xlabel('Temperature (°C)')
    ax2_comp.set_ylabel('Capacitance (fF)')
    ax2_comp.grid(True, linestyle='--', alpha=0.6)
    
    # Indicate fixed Vdc for Compensation with a text box
    if vdc_opt_comp_mark is not None:
        fixed_vdc_text_comp = f"Vdc Fixed @ {vdc_opt_comp_mark*1000:.0f} mV"
        ax2_comp.text(0.05, 0.95, fixed_vdc_text_comp, transform=ax2_comp.transAxes, 
                      fontsize=11, fontweight='bold', color='darkred', verticalalignment='top', 
                      bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='darkred', alpha=0.9))
        ax2_comp.legend(loc='lower right')

fig2.tight_layout()

# ==========================================
# FIGURE 3: Q-Factor vs Temperature
# ==========================================
fig3, (ax3_prop, ax3_comp) = plt.subplots(1, 2, figsize=(12, 5))
fig3.suptitle('Q-Factor vs Temperature', fontsize=14, fontweight='bold')

if not dfs['prop_tb2'].empty and 'Q_factor:ac' in dfs['prop_tb2'].columns:
    ax3_prop.plot(dfs['prop_tb2']['Sweep:Variable:temp'], dfs['prop_tb2']['Q_factor:ac'], color='green', linewidth=2, marker='o')
    ax3_prop.set_title('Proportional + Integral Varactor')
    ax3_prop.set_xlabel('Temperature (°C)')
    ax3_prop.set_ylabel('Q Factor')
    ax3_prop.grid(True, linestyle='--', alpha=0.6)

if not dfs['comp_tb2'].empty and 'Q_factor:ac' in dfs['comp_tb2'].columns:
    t_orig_q = dfs['comp_tb2']['Sweep:Variable:temp']
    plot_df3 = pd.DataFrame({'T': t_orig_q, 'Q': dfs['comp_tb2']['Q_factor:ac']}).sort_values('T')
    ax3_comp.plot(plot_df3['T'], plot_df3['Q'], color='teal', linewidth=2, marker='o')
    ax3_comp.set_title('Compensation Varactor')
    ax3_comp.set_xlabel('Temperature (°C)')
    ax3_comp.set_ylabel('Q Factor')
    ax3_comp.grid(True, linestyle='--', alpha=0.6)
fig3.tight_layout()

# ==========================================
# DATA PROCESSING FOR COMPENSATION AND FIGURE 4
# ==========================================
comp_ready = not dfs['comp_tb1'].empty and cap_col in dfs['comp_tb1'].columns
prop_ready = not dfs['prop_tb2'].empty and cap_col in dfs['prop_tb2'].columns

if comp_ready and prop_ready and not dfs['typ_ct'].empty:
    
    # 1. Get the Typical Uncompensated CT Curve
    df_typ = dfs['typ_ct']
    df_tt = df_typ[df_typ['Corner'] == 'TT25'].copy()
    typ_temps = df_tt['Sweep:Variable:temp']
    typ_caps = get_cap_ff(df_tt, 'Cap_measuredAt1Mhz_M7:ac')
    
    # 2. Calculate the Target Total CT Curve (Mirrored / 1.2)
    target_total_c = typ_caps.values[::-1] / 1.2
    
    # 3. Get the Real C_prop values
    df_ct = dfs['prop_tb2'].copy()
    temps = df_ct['Sweep:Variable:temp']
    
    # MULTIPLYING BY 2 FOR PROPORTIONAL + INTEGRAL
    c_prop_arr = get_cap_ff(df_ct) * 2 
    
    # 4. Calculate mathematically required capacitance (dynamic TARGET)
    c_comp_req = target_total_c - c_prop_arr
    
    # 5. Get available C_comp from the CV Curve (tb1) and setup mapping
    df_cv_clean = dfs['comp_tb1'].copy()
    df_cv_clean['C_comp_fF'] = get_cap_ff(df_cv_clean)
    df_cv_clean['Vdc'] = df_cv_clean['Sweep:Variable:Vdc'].apply(parse_units)
    df_cv_clean = df_cv_clean.dropna(subset=['C_comp_fF', 'Vdc'])
    
    # Sort for robust interpolation
    df_cv_v_sorted = df_cv_clean.sort_values('Vdc')
    df_cv_c_sorted = df_cv_clean.sort_values('C_comp_fF')

    # Map needed capacitance -> voltage (Clamped to prevent extrapolation errors)
    cv_interp = interp1d(
        df_cv_c_sorted['C_comp_fF'], 
        df_cv_c_sorted['Vdc'], 
        bounds_error=False, 
        fill_value=(df_cv_c_sorted['Vdc'].iloc[0], df_cv_c_sorted['Vdc'].iloc[-1])
    )
    
    # Map actual voltage -> provided capacitance (Clamped)
    vc_interp = interp1d(
        df_cv_v_sorted['Vdc'], 
        df_cv_v_sorted['C_comp_fF'], 
        bounds_error=False, 
        fill_value=(df_cv_v_sorted['C_comp_fF'].iloc[0], df_cv_v_sorted['C_comp_fF'].iloc[-1])
    )
    
    # 6. Apply mapping and SATURATION (0.6V to 0.9V)
    vdc_mapped = cv_interp(c_comp_req)
    vdc_actual = np.clip(vdc_mapped, 0.6, 0.9)  # Physical saturation limits
    c_comp_achieved = vc_interp(vdc_actual)
    
    # Total actual capacitance using saturated mapping
    c_total = c_prop_arr + c_comp_achieved

    # ==========================================
    # FIGURE 4: COMPENSATED VS TARGET VS TYPICAL CT CURVE
    # ==========================================
    fig4, ax4 = plt.subplots(figsize=(8, 5))
    
    # Plot Actual Total vs Target vs Uncompensated
    ax4.plot(temps, c_total, color='green', linewidth=3, marker='^', label='Compensated Total (Saturated 0.6V-0.9V)')
    ax4.plot(typ_temps, typ_caps, color='red', linestyle='--', linewidth=2, marker='o', label='Uncompensated Typical CT')
    
    # Plot the dynamic Target Total C curve
    ax4.plot(temps, target_total_c, color='purple', linestyle=':', linewidth=3, alpha=0.9, label="Target Total C (Mirrored / 1.2)")
    
    # Set limits based on the data
    min_y = min(target_total_c.min(), typ_caps.min(), c_total.min()) * 0.95
    max_y = max(target_total_c.max(), typ_caps.max(), c_total.max()) * 1.05
    ax4.set_ylim(min_y, max_y)

    ax4.set_title('Capacitance Stability (with Vdc Saturation)', fontweight='bold')
    ax4.set_xlabel('Temperature (°C)')
    ax4.set_ylabel('Capacitance (fF)')
    ax4.grid(True, linestyle='--', alpha=0.6)
    ax4.legend(loc='best')
    fig4.tight_layout()

plt.show()